# Agentic MCTS: Two-Component Loop Demo

This notebook walks through the **two-component optimization loop** step by step:

```
┌─────────────────────────────────────────────────────────────────────┐
│                      OPTIMIZATION LOOP                            │
│                                                                     │
│  ┌──────────────────────┐        ┌──────────────────────────────┐ │
│  │  Component 1: MCTS   │        │  Component 2: LLM Analyzer  │ │
│  │                      │        │                              │ │
│  │  Play N games with   │ ──────►│  Read gameplay traces +     │ │
│  │  current heuristics  │ traces │  current heuristic code     │ │
│  │                      │        │                              │ │
│  │  Log every MCTS      │        │  Analyze losing patterns    │ │
│  │  decision + board    │        │  Propose improved Python    │ │
│  │  state               │◄────── │  heuristic functions        │ │
│  │                      │new code│                              │ │
│  └──────────────────────┘        └──────────────────────────────┘ │
│                                                                     │
│  Repeat until convergence (win rate stabilizes)                   │
└─────────────────────────────────────────────────────────────────────┘
```

## Key files

| File | Role |
|------|------|
| `mcts/game_interface.py` | Abstract `Game` + `GameState` base (any game) |
| `mcts/node.py` | `MCTSNode` — tree node with UCB1 |
| `mcts/mcts_engine.py` | `MCTSEngine` — core search + pluggable heuristics |
| `mcts/heuristics.py` | **LLM-optimizable** default heuristic functions |
| `mcts/trace_logger.py` | Records game traces for LLM analysis |
| `mcts/games/connect_four.py` | Pure-Python Connect Four |
| `mcp_host.py` | MCP Host for calling Qwen 3.5 |

---
# Part 1: Understanding the MCTS Engine

In [1]:
## Cell 1: Setup — imports & game creation
import sys, random
from mcts import MCTSEngine, TraceLogger
from mcts.games.connect_four import ConnectFour, ConnectFourState
from mcts.node import MCTSNode
from mcts import heuristics as H

game = ConnectFour()
print(f"Game: {game.name()}, Players: {game.num_players()}")

# Show the initial board
state = game.new_initial_state()
print(f"\nInitial state (Player {state.current_player()}'s turn):")
print(state)
print(f"\nLegal actions: {state.legal_actions()}")

Game: Connect Four, Players: 2

Initial state (Player 0's turn):
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
0 1 2 3 4 5 6

Legal actions: [0, 1, 2, 3, 4, 5, 6]


In [2]:
## Cell 2: Inspect default heuristics (the "tools" LLM will optimize)
import inspect

print("="*60)
print("CURRENT HEURISTIC FUNCTIONS (LLM optimization targets)")
print("="*60)

heuristic_fns = [
    ("1. rollout_policy",     H.random_rollout_policy),
    ("2. evaluation",         H.null_evaluation),
    ("3. exploration_weight", H.default_exploration_weight),
    ("4. action_priority",    H.default_action_priority),
]

for label, fn in heuristic_fns:
    print(f"\n--- {label} ---")
    print(inspect.getsource(fn))

CURRENT HEURISTIC FUNCTIONS (LLM optimization targets)

--- 1. rollout_policy ---
def random_rollout_policy(state: GameState) -> Any:
    """
    Choose an action during the simulation (rollout) phase.

    Args:
        state: Current game state (non-terminal, has legal actions).

    Returns:
        One of state.legal_actions().

    Default: uniform random.
    The LLM can replace this with a smarter policy (e.g., prefer centre
    columns in Connect Four, capture moves in Chess, etc.).
    """
    return random.choice(state.legal_actions())


--- 2. evaluation ---
def null_evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Optionally evaluate a non-terminal state and return a value estimate.

    Args:
        state:              Current (possibly non-terminal) game state.
        perspective_player: The player whose perspective the value is from.

    Returns:
        A float in [-1, 1] if you want to short-circuit the rollout,
        or None to l

In [3]:
## Cell 3: Walk through ONE MCTS search step-by-step
## This reveals Selection → Expansion → Simulation → Backprop

engine = MCTSEngine(game, iterations=50)  # small for visibility
state = game.new_initial_state()

# Run search and inspect the tree
best_action = engine.search(state)

# Look at the root node
root = engine._get_node(state)
print(f"Root visits: {root.visits}")
print(f"Root value:  {root.value:.2f}")
print(f"Best action: column {best_action}")

print(f"\nAll children (action → visits / avg_value):")
for action, child in sorted(root.children.items()):
    avg = child.value / max(child.visits, 1)
    bar = '█' * int(child.visits)
    print(f"  col {action}: {child.visits:3d} visits, avg={avg:+.3f}  {bar}")

print(f"\nTransposition table size: {len(engine.transposition_table)} nodes")

Root visits: 50
Root value:  -4.00
Best action: column 3

All children (action → visits / avg_value):
  col 0:   7 visits, avg=-0.143  ███████
  col 1:   7 visits, avg=-0.143  ███████
  col 2:   9 visits, avg=-0.111  █████████
  col 3:  15 visits, avg=+0.200  ███████████████
  col 4:   2 visits, avg=-1.000  ██
  col 5:   2 visits, avg=-1.000  ██
  col 6:   8 visits, avg=+0.000  ████████

Transposition table size: 51 nodes


---
# Part 2: Component 1 — MCTS Plays Games & Collects Traces

In [6]:
## Cell 4: Play games with DEFAULT heuristics (baseline)
## Using very low iterations (10) so MCTS is weak enough to sometimes lose

engine = MCTSEngine(game, iterations=10)

print("Playing 30 games: MCTS (P0, 10 iters) vs Random (P1)...")
print("Using DEFAULT heuristics (random rollout, no evaluation)\n")

baseline_stats = engine.play_many(
    num_games=30,
    mcts_player=0,
    clear_table_each_game=True,
)

print(f"\n{'='*40}")
print(f"BASELINE RESULTS: {baseline_stats}")
print(f"{'='*40}")

Playing 30 games: MCTS (P0, 10 iters) vs Random (P1)...
Using DEFAULT heuristics (random rollout, no evaluation)

  Game 3/30: winner=P0, outcome=[1.0, -1.0]
  Game 6/30: winner=P0, outcome=[1.0, -1.0]
  Game 9/30: winner=P0, outcome=[1.0, -1.0]
  Game 12/30: winner=P0, outcome=[1.0, -1.0]
  Game 15/30: winner=P0, outcome=[1.0, -1.0]
  Game 18/30: winner=P0, outcome=[1.0, -1.0]
  Game 21/30: winner=P1, outcome=[-1.0, 1.0]
  Game 24/30: winner=P0, outcome=[1.0, -1.0]
  Game 27/30: winner=P0, outcome=[1.0, -1.0]
  Game 30/30: winner=P0, outcome=[1.0, -1.0]

BASELINE RESULTS: {'total_games': 30, 'wins': 24, 'losses': 6, 'draws': 0, 'win_rate': 0.8}


In [7]:
## Cell 5: Inspect the gameplay traces
## This is what gets fed to the LLM for analysis

print(f"Total games logged: {len(engine.logger.games)}")
print(f"Losses: {len(engine.logger.losing_games())}")
print()

# Show one game trace in detail
sample_game = engine.logger.games[0]
print(f"--- Game {sample_game.game_id} ---")
print(f"Winner: Player {sample_game.winner}")
print(f"MCTS was: Player {sample_game.mcts_player}")
print(f"Total moves: {sample_game.total_moves}")
print(f"Time: {sample_game.elapsed_sec}s")
print()

# Show first 3 MCTS decisions with board state
for move in sample_game.moves[:3]:
    print(f"Turn {move.turn} | Player {move.player}")
    print(f"  Board:\n{move.state_str}")
    print(f"  Chose: column {move.chosen_action}")
    print(f"  MCTS visits: {move.mcts_visits}")
    print(f"  Top actions: {move.top_actions[:3]}")
    print()

Total games logged: 30
Losses: 6

--- Game 1 ---
Winner: Player 1
MCTS was: Player 0
Total moves: 11
Time: 0.003s

Turn 1 | Player 0
  Board:
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
0 1 2 3 4 5 6
  Chose: column 3
  MCTS visits: 10
  Top actions: [{'action': 3, 'visits': 2, 'avg_value': 1.0}, {'action': 6, 'visits': 2, 'avg_value': 1.0}, {'action': 1, 'visits': 2, 'avg_value': 0.0}]

Turn 2 | Player 0
  Board:
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O . . X . . .
0 1 2 3 4 5 6
  Chose: column 3
  MCTS visits: 10
  Top actions: [{'action': 3, 'visits': 2, 'avg_value': 1.0}, {'action': 1, 'visits': 2, 'avg_value': 1.0}, {'action': 0, 'visits': 2, 'avg_value': 1.0}]

Turn 3 | Player 0
  Board:
0 1 2 3 4 5 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . . X . . .
O . . X . . .
0 1 2 3 4 5 6
  Chose: column 2
  MCTS visits: 10
  Top actions: [{'action': 2, 'visits': 2, 'avg_va

In [8]:
## Cell 6: See the LLM-formatted trace report
## This is the exact string we'll send to Qwen 3.5

llm_report = engine.logger.format_for_llm(max_games=3, losses_only=False)
print(llm_report)
print(f"\n(Report length: {len(llm_report)} chars)")

MCTS GAMEPLAY TRACE  ({'total_games': 30, 'wins': 24, 'losses': 6, 'draws': 0, 'win_rate': 0.8})

--- Game 1 | Winner: Player 1 | MCTS was Player 0 | Moves: 11 | Time: 0.003s ---
  Turn 1: P0 chose action=3 (visits=10, top=[{'action': 3, 'visits': 2, 'avg_value': 1.0}, {'action': 6, 'visits': 2, 'avg_value': 1.0}, {'action': 1, 'visits': 2, 'avg_value': 0.0}])
    Board:
      0 1 2 3 4 5 6
      . . . . . . .
      . . . . . . .
      . . . . . . .
      . . . . . . .
      . . . . . . .
      . . . . . . .
      0 1 2 3 4 5 6
  Turn 2: P0 chose action=3 (visits=10, top=[{'action': 3, 'visits': 2, 'avg_value': 1.0}, {'action': 1, 'visits': 2, 'avg_value': 1.0}, {'action': 0, 'visits': 2, 'avg_value': 1.0}])
    Board:
      0 1 2 3 4 5 6
      . . . . . . .
      . . . . . . .
      . . . . . . .
      . . . . . . .
      . . . . . . .
      O . . X . . .
      0 1 2 3 4 5 6
  Turn 3: P0 chose action=2 (visits=10, top=[{'action': 2, 'visits': 2, 'avg_value': 1.0}, {'action': 6, 'visit

---
# Part 3: Component 2 — LLM Analyzes Traces & Proposes Better Heuristics

In [9]:
## Cell 7: Build the LLM prompt
## This combines: game rules + current heuristic code + gameplay traces

heuristic_source = engine.get_heuristic_source()

SYSTEM_PROMPT = """You are an expert game AI engineer. You optimize MCTS heuristic functions 
for board games. You analyze gameplay traces to find weaknesses and write improved Python code.

RULES:
- You must output valid Python function(s) that can replace the current heuristics.
- Each function must match the exact signature shown.
- Wrap each function in a ```python code block.
- Focus on the most impactful improvement first."""

USER_PROMPT = f"""## Game: Connect Four (6×7 board, 4-in-a-row to win)
Players: 0 (X) and 1 (O). MCTS controls Player 0.

## Current Heuristic Functions

### rollout_policy(state) -> action
```python
{heuristic_source['rollout_policy']}
```

### evaluation(state, perspective_player) -> float | None
```python
{heuristic_source['evaluation']}
```

### exploration_weight(root_visits) -> float
```python
{heuristic_source['exploration_weight']}
```

### action_priority(state, actions) -> list[action]
```python
{heuristic_source['action_priority']}
```

## Gameplay Traces (MCTS = Player 0 vs Random = Player 1)
{llm_report}

## Task
Analyze the gameplay traces. Identify weaknesses in the current heuristics.
Write ONE improved Python function (choose the most impactful heuristic to improve).
Explain your reasoning, then provide the code.

IMPORTANT: The state object is a ConnectFourState with these methods:
- state.legal_actions() -> list[int]  (columns 0-6)
- state.current_player() -> int  (0 or 1)
- state.get_piece(row, col) -> 0 | 1 | None
- state.column_height(col) -> int
- state.clone() -> ConnectFourState
- state.apply_action(col) -> None
- state.is_terminal() -> bool
- state.returns() -> list[float]  (+1 win, -1 loss, 0 draw)
Board: 6 rows (0=top) × 7 cols. Pieces drop to the lowest empty row."""

print("SYSTEM PROMPT:")
print(SYSTEM_PROMPT)
print(f"\nUSER PROMPT ({len(USER_PROMPT)} chars):")
print(USER_PROMPT[:500])
print("...")

SYSTEM PROMPT:
You are an expert game AI engineer. You optimize MCTS heuristic functions 
for board games. You analyze gameplay traces to find weaknesses and write improved Python code.

RULES:
- You must output valid Python function(s) that can replace the current heuristics.
- Each function must match the exact signature shown.
- Wrap each function in a ```python code block.
- Focus on the most impactful improvement first.

USER PROMPT (14346 chars):
## Game: Connect Four (6×7 board, 4-in-a-row to win)
Players: 0 (X) and 1 (O). MCTS controls Player 0.

## Current Heuristic Functions

### rollout_policy(state) -> action
```python
def random_rollout_policy(state: GameState) -> Any:
    """
    Choose an action during the simulation (rollout) phase.

    Args:
        state: Current game state (non-terminal, has legal actions).

    Returns:
        One of state.legal_actions().

    Default: uniform random.
    The LLM can replace this with a 
...


In [12]:
## Cell 8: Call Qwen 3.5 to analyze traces and propose improvements
import requests, json

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3.5:35b"

print(f"Calling {MODEL} for heuristic optimization...")
print("(This may take 30-60 seconds)\n")

resp = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": USER_PROMPT},
        ],
        "stream": False,
        "options": {"temperature": 0.3, "num_predict": 4096},
    },
    timeout=300,
)
resp.raise_for_status()

msg = resp.json()["message"]
llm_response = msg.get("content", "")
thinking = msg.get("thinking", "")

if thinking:
    print("💭 LLM Thinking (first 800 chars):")
    print(thinking[:800])
    print("...\n")

# Qwen 3.5 sometimes puts code blocks in 'thinking' and leaves 'content' empty.
# Combine both for code extraction.
llm_full_text = llm_response + "\n" + thinking

print("🤖 LLM Response (content):")
if llm_response.strip():
    print(llm_response)
else:
    print("(empty — code is likely in the thinking field)")
    print("\n🔍 Searching thinking field for code blocks...")

Calling qwen3.5:35b for heuristic optimization...
(This may take 30-60 seconds)

💭 LLM Thinking (first 800 chars):
The user wants me to analyze the gameplay traces of Connect Four and identify weaknesses in the current MCTS heuristics. Then I need to write ONE improved Python function that addresses the most impactful weakness.

Looking at the traces:

1. **Game 1**: MCTS (Player 0) lost despite having 24 wins out of 30 games. The game lasted 11 moves. Looking at the board state, Player 0 made several moves that didn't seem optimal. At Turn 8, Player 0 played action=2 which created a vertical line of X's but also allowed Player 1 to potentially create threats.

2. **Game 2**: MCTS won in 10 moves. The play was more aggressive and strategic.

3. **Game 3**: MCTS won in 9 moves. Similar pattern to Game 2.

**Key Observations from the traces:**

1. **Rollout Policy is purely random**: The `random_rollout_
...

🤖 LLM Response (content):
## Analysis of Gameplay Traces

### Key Weaknesses Id

In [14]:
## Cell 9: Extract and execute the LLM's proposed code
import re
from mcts.game_interface import GameState
from mcts.games.connect_four import ConnectFourState

# Extract Python code blocks from combined LLM output (content + thinking)
code_blocks = re.findall(r'```python\n(.*?)```', llm_full_text, re.DOTALL)

if not code_blocks:
    print("⚠️ No Python code blocks found in LLM response.")
    print("You may need to re-run Cell 8 or manually write the function.")
else:
    print(f"Found {len(code_blocks)} code block(s):\n")
    for i, block in enumerate(code_blocks):
        print(f"--- Block {i+1} ---")
        print(block)
        print()
    
    # Execute the code to define the function(s)
    # Provide all types the LLM might reference in its code
    combined_code = "\n".join(code_blocks)
    exec_namespace = {
        "random": random,
        "math": __import__("math"),
        "GameState": GameState,
        "ConnectFourState": ConnectFourState,
    }
    exec(combined_code, exec_namespace)
    
    # Detect which functions were defined
    heuristic_names = ["rollout_policy", "evaluation", "exploration_weight", "action_priority"]
    # Also accept common variants the LLM might use
    name_map = {
        "random_rollout_policy": "rollout_policy",
        "null_evaluation": "evaluation",
        "default_exploration_weight": "exploration_weight",
        "default_action_priority": "action_priority",
        "improved_rollout_policy": "rollout_policy",
        "improved_evaluation": "evaluation",
        "smart_rollout_policy": "rollout_policy",
        "center_bias_rollout": "rollout_policy",
    }
    
    builtin_keys = {"random", "math", "GameState", "ConnectFourState",
                    "__builtins__", "Any", "Optional"}
    new_fns = {}
    for key, val in exec_namespace.items():
        if callable(val) and key not in builtin_keys:
            target = name_map.get(key, key)
            if target in heuristic_names:
                new_fns[target] = val
    
    print(f"\n✅ Extracted functions: {list(new_fns.keys())}")

Found 1 code block(s):

--- Block 1 ---
def evaluation(state: ConnectFourState, perspective_player: int) -> float | None:
    """
    Evaluate Connect Four board position from perspective_player's view.
    
    Args:
        state: Current game state.
        perspective_player: Player whose perspective the value is from (0 or 1).
    
    Returns:
        A float in [-1, 1] representing position value, or None to continue rollout.
    """
    # Check if terminal state - return immediate result
    if state.is_terminal():
        returns = state.returns()
        if perspective_player == 0:
            return returns[0]
        else:
            return returns[1]
    
    # Get board dimensions
    rows, cols = 6, 7
    board = [[state.get_piece(r, c) for c in range(cols)] for r in range(rows)]
    opponent = 1 - perspective_player
    
    # Initialize score
    score = 0.0
    
    # Helper to count pieces in a line
    def count_pieces(line, player):
        return sum(1 for p in l

---
# Part 4: Hot-Swap & Re-Evaluate

In [15]:
## Cell 10: Hot-swap the improved heuristic(s) into the MCTS engine

# Create a FRESH engine with the same settings
improved_engine = MCTSEngine(game, iterations=10)

# Swap in the LLM-proposed functions
for name, fn in new_fns.items():
    improved_engine.set_heuristic(name, fn)
    print(f"✅ Swapped: {name}")

print(f"\nPlaying 30 games with IMPROVED heuristics...\n")

improved_stats = improved_engine.play_many(
    num_games=30,
    mcts_player=0,
    clear_table_each_game=True,
)

print(f"\n{'='*50}")
print(f"BASELINE:  {baseline_stats}")
print(f"IMPROVED:  {improved_stats}")
print(f"{'='*50}")

delta = improved_stats['win_rate'] - baseline_stats['win_rate']
print(f"\nWin rate change: {delta:+.1%}")

✅ Swapped: evaluation

Playing 30 games with IMPROVED heuristics...

  Game 3/30: winner=P0, outcome=[1.0, -1.0]
  Game 6/30: winner=P0, outcome=[1.0, -1.0]
  Game 9/30: winner=P0, outcome=[1.0, -1.0]
  Game 12/30: winner=P0, outcome=[1.0, -1.0]
  Game 15/30: winner=P0, outcome=[1.0, -1.0]
  Game 18/30: winner=P0, outcome=[1.0, -1.0]
  Game 21/30: winner=P0, outcome=[1.0, -1.0]
  Game 24/30: winner=P0, outcome=[1.0, -1.0]
  Game 27/30: winner=P0, outcome=[1.0, -1.0]
  Game 30/30: winner=P0, outcome=[1.0, -1.0]

BASELINE:  {'total_games': 30, 'wins': 24, 'losses': 6, 'draws': 0, 'win_rate': 0.8}
IMPROVED:  {'total_games': 30, 'wins': 29, 'losses': 1, 'draws': 0, 'win_rate': 0.9667}

Win rate change: +16.7%
